# WHOMOLLM Project
## Geo context prompt (Version 3) 28.Jan.2026
1. Raw data
2. stay segmentation
3. reverse geocoding
4. top 5 POIs aggregation
5. semantic stay table
6. LLM prompt (daily narrative/ activity inference/ other surplus info)

## 1. Test GPU, model 

In [1]:
import os, json, re, subprocess, sys
import torch
import requests
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    p = torch.cuda.get_device_properties(0)
    print("VRAM (GB):", round(p.total_memory/1024**3, 2))

# Check ollama is reachable
r = requests.get("http://127.0.0.1:11434/api/tags", timeout=3)
print("Ollama OK. Models:", [m["name"] for m in r.json().get("models",[])])

KeyboardInterrupt: 

## 2. Generate prompts (3nd version with just geo context information and behaviour info, predict household size and household income level ... for all users)
Version control:
This is the basic prompt based on geocontext and behaviour info, version 3.

In [ ]:
#   ---------------------------
# Version 3 (28 Jan 2026) prompt with geo context to predict features
#   ---------------------------

import os, re, json
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import pdb

# ---------------------------
# Config
# ---------------------------
RANDOM_SEED = 42
N_USERS     = 5
MAX_EVENTS  = 300
HOUR_BIN    = 1
PREC        = int(os.getenv("COORD_PREC", "4"))


DATA_SP     = Path("/data/baliu/python_code/data/sp_all copy.csv")
PROMPTS_OUT = Path("/data/baliu/python_code/data/prompts_v3_features_28Jan2026.json")

# -------------------------------------------   
# Load (strings) + base cleaning
# -------------------------------------------
sp = pd.read_csv(DATA_SP, dtype=str, engine="python", on_bad_lines="skip")
print("sp shape:", sp.shape)
# print("sp columns:", sp.columns.tolist())

# Datetimes
sp["started_at"]  = pd.to_datetime(sp["started_at"], errors="coerce", utc=True)
sp["finished_at"] = pd.to_datetime(sp.get("finished_at"), errors="coerce", utc=True)

# Drop unusable
sp = sp.dropna(subset=["user_id", "started_at", "location_id"]).copy()
sp["user_id"]     = sp["user_id"].astype(str)
sp["location_id"] = sp["location_id"].astype(str)

#sp["duration_min"] = ((sp["finished_at"] - sp["started_at"]).dt.total_seconds() / 60.0).astype(float)
print(sp["duration"].head())
print(sp["duration"].dtype)

# Force numeric
sp["act_duration"] = pd.to_numeric(
    sp["act_duration"],
    errors="coerce"
)

# change it to hours
sp["act_duration_h"] = (sp["act_duration"] / 60.0).round(1)
print(sp["act_duration_h"].head())
print(sp["act_duration_h"].dtype)

# length_km: convert to numeric, coerce errors to NaN, then divide by 1000
sp["length_km"] = pd.to_numeric(sp["length"], errors="coerce") / 1000.0
sp["length_km"] = sp["length_km"].round(1)

# Time features
sp["date"]     = sp["started_at"].dt.date
sp["dow"]      = sp["started_at"].dt.dayofweek.astype(int)        # 0=Mon
sp["hour_bin"] = (sp["started_at"].dt.hour // HOUR_BIN * HOUR_BIN).astype(int)

# ---------------------------
# Format from wkt point -> lon/lat (rounded)
# ---------------------------
WKT_POINT_RE = re.compile(
    r"POINT\s*\(\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s+([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)\s*\)",
    re.I,
)

def extract_lonlat_from_wkt(s: str):
    m = WKT_POINT_RE.search(str(s))
    if not m: return (np.nan, np.nan)
    return float(m.group(1)), float(m.group(2))

sp["geometry"] = sp.get("geometry", pd.Series(index=sp.index)).astype(str)
lonlat = sp["geometry"].apply(extract_lonlat_from_wkt)
sp[["lon","lat"]] = pd.DataFrame(lonlat.tolist(), index=sp.index)
sp["lon"] = pd.to_numeric(sp["lon"], errors="coerce").round(PREC)
sp["lat"] = pd.to_numeric(sp["lat"], errors="coerce").round(PREC)

# Lightweight tags
sp["geometry_type"] = np.where(sp["lon"].notna() & sp["lat"].notna(), "point", "-")
if "mode" not in sp.columns: sp["mode"] = "-"

sp shape: (1152987, 10)
0    1093.0
1      39.0
2      75.0
3     164.0
4      36.0
Name: duration, dtype: object
object
0    18.2
1     1.2
2     1.4
3     5.6
4     1.2
Name: act_duration_h, dtype: float64
float64


In [ ]:
print(sp["act_duration_h"].head())
print(sp["act_duration_h"].dtype)
print(sp["length_km"].head())
print(sp["length_km"].dtype)

0    18.2
1     1.2
2     1.4
3     5.6
4     1.2
Name: act_duration_h, dtype: float64
float64
0     0.0
1    11.6
2     2.1
3     4.8
4    12.6
Name: length_km, dtype: float64
float64


In [ ]:
print ("sp after cleaning:", sp.shape)
print (sp.head(5))

sp after cleaning: (862871, 18)
  id user_id                       started_at  \
0  0   AAGAF 2019-10-09 11:30:34.141000+00:00   
1  1   AAGAF 2019-10-10 06:14:49.141999+00:00   
2  2   AAGAF 2019-10-10 07:03:24.426000+00:00   
3  3   AAGAF 2019-10-10 11:10:24.605999+00:00   
4  4   AAGAF 2019-10-10 14:30:45.187999+00:00   

                       finished_at location_id  mode              length  \
0 2019-10-10 05:43:17.674999+00:00           0   Car                 0.0   
1 2019-10-10 06:53:54.841000+00:00           1   Car  11615.408548376423   
2 2019-10-10 08:18:20.864000+00:00           2   Car  2104.8558581266784   
3 2019-10-10 13:54:34.799339+00:00           2  Walk   4847.706521391238   
4 2019-10-10 15:07:07.127239+00:00           3   Car  12621.909934521313   

                                       geometry duration  act_duration  \
0  POINT (7.565219245342489 47.545616372908086)   1093.0        1093.0   
1  POINT (7.5637597959179645 47.54794767256367)     39.0          71

In [ ]:
# ---------------------------
# Sample 1 week per user
# ---------------------------
def sample_one_week_per_user(sp, seed=42):
    rng = np.random.default_rng(seed)
    sp = sp.copy()
    sp["date"] = pd.to_datetime(sp["date"])

    out = []
    for uid, df_u in sp.groupby("user_id"):
        days = (
            df_u[["date"]]
            .drop_duplicates()
            .sort_values("date")
            .reset_index(drop=True)
        )

        if len(days) < 7:
            continue
        days["delta"] = days["date"].diff().dt.days.fillna(1).astype(int)
        days["block"] = (days["delta"] >1).cumsum()

        valid_blocks = (
            days.groupby("block")
            .filter(lambda x: len(x) >=7)
        )

        if valid_blocks.empty:
            continue
        candidate_starts = []
        for _, g in valid_blocks.groupby("block"):
            block_dates = g["date"].sort_values().reset_index(drop=True)
            for i in range(len(block_dates) - 6):
                candidate_starts.append(g.iloc[i]["date"])

        start_date = rng.choice(candidate_starts)
        week_dates = pd.date_range(start=start_date, periods=7, freq= 'D')

        out.append(
            df_u[df_u["date"].isin(week_dates)]

        )
       
    return pd.concat(out, ignore_index=True)

sp_sampled2 = sample_one_week_per_user(sp, seed=RANDOM_SEED)
print("sp_sampled2 shape:", sp_sampled2.shape)
print(sp_sampled2.head(2))

# ---------------------------
# save sampled data
# ---------------------------
SP_SAMPLED2_OUT = Path("/data/baliu/python_code/data/sp_sampled2_1week_28Jan2026.csv")
sp_sampled2.to_csv(SP_SAMPLED2_OUT, index=False)
print("Saved sp_sampled2_1week_28Jan2026 to", SP_SAMPLED2_OUT)  

sp_sampled2 shape: (60738, 18)
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   

                       finished_at location_id  mode             length  \
0 2019-10-23 03:56:33.583344+00:00          10   Bus  3823.055092733663   
1 2019-10-23 08:31:59.036438+00:00          16  Tram  8232.492568529495   

                                       geometry duration  act_duration  \
0  POINT (7.564359990583688 47.546459921667946)   1097.0        6456.0   
1   POINT (7.603269084785703 47.58100341208616)    116.0         275.0   

   act_duration_h  length_km       date  dow  hour_bin     lon      lat  \
0           107.6        3.8 2019-10-22    1         9  7.5644  47.5465   
1             4.6        8.2 2019-10-23    2         6  7.6033  47.5810   

  geometry_type  
0         point  
1         point  
Saved sp_sampled2_1week_28Jan2026 to /data/baliu/python_code/data/sp_sampled2_1week_28Jan

#### Load location name and POIs top 5

In [ ]:
# 1. Load toponym data from shepefiles
import os, re, json
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import pdb
import geopandas as gpd
import fiona
# read parquet file
import pyarrow.parquet as pq
 
CACHE_DIR   = Path("/data/baliu/python_code/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
NOM_CACHE_PATH = CACHE_DIR / "nominatim_cache.parquet"
nom = pd.read_parquet(NOM_CACHE_PATH)
print("Nominatim cache loaded:", nom.shape)
print("Nominatim columns:", nom.columns.tolist())
nom.head(30)
groupby_category = nom.groupby('city').size()
print(groupby_category) 

Nominatim cache loaded: (370983, 6)
Nominatim columns: ['lon', 'lat', 'road', 'neighbourhood', 'city', 'postcode']
city
A Coruña                 2
A Illa de Arousa         2
A Pobra do Caramiñal     7
A Seara                  1
Aachen                   2
                        ..
دهستان سروستان           1
دهستان فجر               3
دهستان کرکس              1
บ้านอ่าวน้ำเมา           1
เทศบาลนครเกาะสมุย       55
Length: 7400, dtype: int64


In [ ]:
from notebook_utils.geocontext import attach_nominatim, clean_nominatim_fields

sp_sampled2 = attach_nominatim(sp_sampled2, nom)
sp_sampled2 = clean_nominatim_fields(sp_sampled2)
print("sp_sampled2 after attaching nominatim:", sp_sampled2.shape)
print("sp_sampled2 columns:", sp_sampled2.columns.tolist())
print(sp_sampled2.head(10))


sp_sampled2 after attaching nominatim: (60738, 22)
sp_sampled2 columns: ['id', 'user_id', 'started_at', 'finished_at', 'location_id', 'mode', 'length', 'geometry', 'duration', 'act_duration', 'act_duration_h', 'length_km', 'date', 'dow', 'hour_bin', 'lon', 'lat', 'geometry_type', 'road', 'neighbourhood', 'city', 'postcode']
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   
2  25   AAGAF 2019-10-23 09:02:58.773999+00:00   
3  26   AAGAF 2019-10-23 09:53:18.151999+00:00   
4  27   AAGAF 2019-10-23 16:44:43.608999+00:00   
5  28   AAGAF 2019-10-24 06:05:44.141999+00:00   
6  29   AAGAF 2019-10-24 07:22:44.510999+00:00   
7  30   AAGAF 2019-10-24 09:55:47.243000+00:00   
8  31   AAGAF 2019-10-24 13:05:05.668999+00:00   
9  32   AAGAF 2019-10-24 15:03:21.792999+00:00   

                       finished_at location_id  mode              length  \
0 2019-10-23 03:56:33.583344+00:00          10 

#### Build token with geocontext info

In [ ]:
from pathlib import Path
from notebook_utils.geocontext import load_poi_frame

POI_PATH = Path("/data/baliu/python_code/data/version2/data/final_pois_nob.gpkg")
poi = load_poi_frame(POI_PATH)
pois = poi
poi.head()


,id,code,name,category,geometry
0,0,3100,Temple de Saint-Jean,Civic,POINT (2498816.948 1117839.539)
1,1,3100,Kapelle Oberes Heiliges Kreuz,Civic,POINT (2691857.357 1192677.631)
2,2,3102,Kirche St. Johannes,Civic,POINT (2599659.594 1202970.041)
3,3,3102,Paroisse catholique,Civic,POINT (2501879.911 1126418.086)
4,4,3300,Mosquée du Petit-Saconnex,Civic,POINT (2498411.634 1119984.893)


In [ ]:
poi = poi.copy()
print(poi["category"].value_counts())
print(f"poi crs: {poi.crs}")


category
Others            158579
Unknown           114894
Entertainment     106734
Shopping           54634
Residential        45556
Transportation     40997
Services            9525
Schools             2904
Civic                780
Name: count, dtype: int64
poi crs: EPSG:2056


In [ ]:
from notebook_utils.geocontext import load_poi_frame

def load_pois_once():
    global poi
    if poi is None:
        print("Loading POIs into memory...")
        poi = load_poi_frame(POI_PATH)
    return poi

print(poi.head(2))


   id  code                           name category  \
0   0  3100           Temple de Saint-Jean    Civic   
1   1  3100  Kapelle Oberes Heiliges Kreuz    Civic   

                          geometry  
0  POINT (2498816.948 1117839.539)  
1  POINT (2691857.357 1192677.631)  


In [ ]:
from notebook_utils.geocontext import build_location_gdf

loc = build_location_gdf(sp_sampled2)


### update the table with places name and pois

update

In [ ]:
# update: 
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree

def bearing_to_direction(dx, dy):
    angle = np.degrees(np.arctan2(dx, dy))
    angle = (angle + 360) % 360

    if 22.5 <= angle < 67.5:
        return "North-East"
    elif 67.5 <= angle < 112.5:
        return "East"
    elif 112.5 <= angle < 157.5:
        return "South-East"
    elif 157.5 <= angle < 202.5:
        return "South"
    elif 202.5 <= angle < 247.5:
        return "South-West"
    elif 247.5 <= angle < 292.5:
        return "West"
    elif 292.5 <= angle < 337.5:
        return "North-West"
    else:
        return "North"

def get_poi_context_for_prompt(sp_sampled2, poi, top_k=5):

    # ---- prepare GeoDataFrames ----
    loc = gpd.GeoDataFrame(
        sp_sampled2.copy(),
        geometry=gpd.points_from_xy(
            sp_sampled2["lon"].astype(float),
            sp_sampled2["lat"].astype(float)
        ),
        crs="EPSG:4326"
    ).to_crs(2056)

    poi = poi.copy().set_crs(epsg=2056, allow_override=True)

    # ---- KDTree on POIs ----
    poi_xy = np.c_[poi.geometry.x.values, poi.geometry.y.values]
    tree = cKDTree(poi_xy)

    loc_xy = np.c_[loc.geometry.x.values, loc.geometry.y.values]
    dists, idxs = tree.query(loc_xy, k=top_k * 5) # get more to filter later

    # ---- build kNN table ----
    rows = []
    for i, (ds, js) in enumerate(zip(dists, idxs)):
        for d, j in zip(ds, js):
            rows.append({
                "location_id": loc.iloc[i]["location_id"],
                "name": poi.iloc[j]["name"],
                "category": poi.iloc[j]["category"],
                "addr_poi_dist_m": d,
                "dx": poi.geometry.iloc[j].x - loc.geometry.iloc[i].x,
                "dy": poi.geometry.iloc[j].y - loc.geometry.iloc[i].y
            })

    joined = pd.DataFrame(rows)

    # ---- clean categories ----
    joined = joined[
        joined["category"].notna() &
        (~joined["category"].str.lower().isin(["unknown", "others"]))
    ]

    # ---- distance + direction ----
    joined["addr_poi_dist_km"] = (joined["addr_poi_dist_m"] / 1000).round(3)
    joined["direction"] = [
        bearing_to_direction(dx, dy)
        for dx, dy in zip(joined.dx, joined.dy)
    ]
    
    joined = (
        joined
        .sort_values("addr_poi_dist_m")
        .groupby("location_id", group_keys=False)
        .head(top_k)
    )

    return joined[
        ["location_id", "name", "category", "addr_poi_dist_km", "direction"]
    ]

def clean_text_part(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.lower() in ["none", "nan", "-", ""]:
        return None
    return s

def build_poi_prompt_text(df):
    out = []
    for _, r in df.iterrows():
        dist = r["addr_poi_dist_km"]
        direction = r["direction"].lower()
        
       
        category = clean_text_part(r.get("category")) or "point of interest"
        name = clean_text_part(r.get("name"))

        if name:
            poi_desc = f"{category} {name}"
        else:
            poi_desc = f"{category}"
        out.append(
            f"{dist} km to the {direction}: {poi_desc}"
        )
    return out

poi_prompt_df = (
    get_poi_context_for_prompt(sp_sampled2, poi, top_k=5)
    .groupby("location_id", group_keys=False)
    .apply(build_poi_prompt_text, include_groups=False)
    .reset_index(name="nearby_places")
)

sp_sampled2 = sp_sampled2.merge(
    poi_prompt_df,
    on="location_id",
    how="left"
)
print(
    get_poi_context_for_prompt(sp_sampled2, poi)
    ["addr_poi_dist_km"]
    .describe()
)

from pathlib import Path

SP_SAMPLED2_OUT = Path("/data/baliu/python_code/data/sp_sampled2_v3_1week_29Jan2026_with_geocontext.csv")
sp_sampled2.to_csv(SP_SAMPLED2_OUT, index=False)

print("Saved sp_sampled2 to", SP_SAMPLED2_OUT)
print(sp_sampled2.head(5))

count    116301.000000
mean        288.675402
std        1458.250814
min           0.000000
25%           0.042000
50%           0.101000
75%           0.262000
max       14837.215000
Name: addr_poi_dist_km, dtype: float64
Saved sp_sampled2 to /data/baliu/python_code/data/sp_sampled2_v3_1week_29Jan2026_with_geocontext.csv
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   
2  25   AAGAF 2019-10-23 09:02:58.773999+00:00   
3  26   AAGAF 2019-10-23 09:53:18.151999+00:00   
4  27   AAGAF 2019-10-23 16:44:43.608999+00:00   

                       finished_at location_id  mode              length  \
0 2019-10-23 03:56:33.583344+00:00          10   Bus   3823.055092733663   
1 2019-10-23 08:31:59.036438+00:00          16  Tram   8232.492568529495   
2 2019-10-23 09:38:13.596999+00:00          17  Walk   2892.684874565825   
3 2019-10-23 16:19:19.816718+00:00          10   Car  3283.143616000176

In [ ]:
# update: 
import numpy as np

def bearing_to_direction(dx, dy):
    angle = np.degrees(np.arctan2(dx, dy))
    angle = (angle + 360) % 360

    if 22.5 <= angle < 67.5:
        return "North-East"
    elif 67.5 <= angle < 112.5:
        return "East"
    elif 112.5 <= angle < 157.5:
        return "South-East"
    elif 157.5 <= angle < 202.5:
        return "South"
    elif 202.5 <= angle < 247.5:
        return "South-West"
    elif 247.5 <= angle < 292.5:
        return "West"
    elif 292.5 <= angle < 337.5:
        return "North-West"
    else:
        return "North"

import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.spatial import cKDTree

def get_poi_context_for_prompt(sp_sampled2, poi, top_k=5):

    # ---- prepare GeoDataFrames ----
    loc = gpd.GeoDataFrame(
        sp_sampled2.copy(),
        geometry=gpd.points_from_xy(
            sp_sampled2["lon"].astype(float),
            sp_sampled2["lat"].astype(float)
        ),
        crs="EPSG:4326"
    ).to_crs(2056)

    poi = poi.copy().set_crs(epsg=2056, allow_override=True)

    # ---- KDTree on pois ----
    poi_xy = np.c_[poi.geometry.x.values, poi.geometry.y.values]
    tree = cKDTree(poi_xy)

    loc_xy = np.c_[loc.geometry.x.values, loc.geometry.y.values]
    dists, idxs = tree.query(loc_xy, k=top_k * 3) # get more to filter later

    # ---- build kNN table ----
    rows = []
    for i, (ds, js) in enumerate(zip(dists, idxs)):
        for d, j in zip(ds, js):
            rows.append({
                "location_id": loc.iloc[i]["location_id"],
                "name": poi.iloc[j]["name"],
                "category": poi.iloc[j]["category"],
                "addr_poi_dist_m": d,
                "dx": poi.geometry.iloc[j].x - loc.geometry.iloc[i].x,
                "dy": poi.geometry.iloc[j].y - loc.geometry.iloc[i].y
            })

    joined = pd.DataFrame(rows)

    # ---- clean categories ----
    joined = joined[
        joined["category"].notna() &
        (~joined["category"].str.lower().isin(["unknown", "others"]))
    ]

    # ---- distance + direction ----
    joined["addr_poi_dist_km"] = (joined["addr_poi_dist_m"] / 1000).round(3)
    joined["direction"] = [
        bearing_to_direction(dx, dy)
        for dx, dy in zip(joined.dx, joined.dy)
    ]
    
    joined = (
        joined
        .sort_values("addr_poi_dist_m")
        .groupby("location_id", group_keys=False)
        .head(top_k)
    )

    return joined[
        ["location_id", "name", "category", "addr_poi_dist_km", "direction"]
    ]

def build_poi_prompt_text(df):
    out = []
    for _, r in df.iterrows():
        name = r["name"] if pd.notna(r["name"]) else "unknown place"
        category = r["category"] if pd.notna(r["category"]) else "point of interest"
        out.append(
            f"{r['addr_poi_dist_km']} km to the {r['direction'].lower()}: {category} {name}"
        )
    return "; ".join(out)

poi_prompt_df = (
    get_poi_context_for_prompt(sp_sampled2, poi, top_k=5)
    .groupby("location_id", group_keys=False)
    .apply(build_poi_prompt_text, include_groups=False)
    .reset_index(name="nearby_places")
)

sp_sampled2 = sp_sampled2.merge(
    poi_prompt_df,
    on="location_id",
    how="left"
)
print(
    get_poi_context_for_prompt(sp_sampled2, poi)
    ["addr_poi_dist_km"]
    .describe()
)

print("nearby_places in columns",
      "nearby_places" in sp_sampled2.columns)

print(
    sp_sampled2["nearby_places"]
    .notna()
    .value_counts(dropna=False)
)

print(sp_sampled2[["location_id", "nearby_places"]].head(5))


from pathlib import Path

SP_SAMPLED2_OUT = Path("/data/baliu/python_code/data/sp_sampled2_v3_with_geocontext.csv")
sp_sampled2.to_csv(SP_SAMPLED2_OUT, index=False)

print("Saved sp_sampled2 to", SP_SAMPLED2_OUT)
print(sp_sampled2.head(5))

count    226778.000000
mean        296.192856
std        1476.196361
min           0.000000
25%           0.059000
50%           0.133000
75%           0.355000
max       14837.298000
Name: addr_poi_dist_km, dtype: float64
nearby_places in columns? True
nearby_places
True     60319
False      419
Name: count, dtype: int64
  location_id                                      nearby_places
0          10  0.004 km to the east: Shopping Tabaklädeli Neu...
1          16  0.086 km to the south-east: Shopping unknown p...
2          17  0.023 km to the west: Entertainment unknown pl...
3          10  0.004 km to the east: Shopping Tabaklädeli Neu...
4           4  1.748 km to the south-east: Entertainment Ökos...
Saved sp_sampled1 to /data/baliu/python_code/data/sp_sampled1_v3_with_geocontext.csv
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   
2  25   AAGAF 2019-10-23 09:02:58.773999+00:00   
3

same as above, upgraded 

In [ ]:
from notebook_utils.geocontext import print_poi_feature_summary

print_poi_feature_summary(sp_sampled2)



Final sp_sampled2 with POI features:
Columns: ['id', 'user_id', 'started_at', 'finished_at', 'location_id', 'mode', 'length', 'geometry', 'duration', 'act_duration', 'act_duration_h', 'length_km', 'date', 'dow', 'hour_bin', 'lon', 'lat', 'geometry_type', 'road', 'neighbourhood', 'city', 'postcode', 'nearby_places_x', 'nearby_places_y', 'nearby_places']

Sample rows:
   id user_id                       started_at  \
0  23   AAGAF 2019-10-22 09:39:28.773999+00:00   
1  24   AAGAF 2019-10-23 06:35:32.236999+00:00   
2  25   AAGAF 2019-10-23 09:02:58.773999+00:00   
3  26   AAGAF 2019-10-23 09:53:18.151999+00:00   
4  27   AAGAF 2019-10-23 16:44:43.608999+00:00   

                       finished_at location_id  mode              length  \
0 2019-10-23 03:56:33.583344+00:00          10   Bus   3823.055092733663   
1 2019-10-23 08:31:59.036438+00:00          16  Tram   8232.492568529495   
2 2019-10-23 09:38:13.596999+00:00          17  Walk   2892.684874565825   
3 2019-10-23 16:19:19.816

In [ ]:
import ast
import pandas as pd

def tokens_compact_1week(
    df_u: pd.DataFrame,
    max_events=50,
    prec=4
) -> list[str]:
    """
    Build natural-language mobility tokens from up to one week of staypoints,
    inserting a date header at the beginning of each day.
    """

    # --------------------------------------------------
    # 1. detect duration column
    # --------------------------------------------------
    duration_col = None
    for c in df_u.columns:
        if c.lower() in ["duration", "duration_min", "act_duration", "act_duration_min", "dur_min"]:
            duration_col = c
            break

    # --------------------------------------------------
    # 2. select usable columns
    # --------------------------------------------------
    use_cols = [
        "started_at", "dow", "hour_bin", "location_id",
        "lon", "lat", "mode",
        "city", "neighbourhood", "road", "nearby_places",
        "postcode"
    ]
    if duration_col:
        use_cols.append(duration_col)

    df = df_u.loc[:, [c for c in use_cols if c in df_u.columns]].copy()

    # --------------------------------------------------
    # 3. datetime handling + ordering + safety cap
    # --------------------------------------------------
    df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
    df = df.dropna(subset=["started_at"]).sort_values("started_at")

    if len(df) > max_events:
        df = df.head(max_events)

    # --------------------------------------------------
    # 4. build tokens with date headers
    # --------------------------------------------------
    toks = []
    current_date = None

    for r in df.itertuples(index=False):
        t = r.started_at
        date_str = t.date().isoformat()

        # ---- insert date header when date changes ----
        if date_str != current_date:
            toks.append(f"Date: {date_str}")
            current_date = date_str

        # ---- time info ----
        hhmm = t.strftime("%H:%M")
        dow = int(getattr(r, "dow", -1))
        hb  = int(getattr(r, "hour_bin", -1))

        # ---- location ----
        lat = round(float(getattr(r, "lat", 0.0)), prec)
        lon = round(float(getattr(r, "lon", 0.0)), prec)

        def clean_addr_part(s):
            if pd.isna(s):
                return None
            s = str(s).strip()
            if s.lower() in ["nan", "none", "-", ""]:
                return None
            return s
        addr_parts = [
            clean_addr_part(getattr(r, "road", None)),
            clean_addr_part(getattr(r, "neighbourhood", None)),
            clean_addr_part(getattr(r, "city", None)),
        ]
        addr_parts = [p for p in addr_parts if p is not None]
        addr = ", ".join(addr_parts) if addr_parts else "unknown address"
        # add postcode
        postcode = clean_addr_part(getattr(r, "postcode", None))
        if postcode is not None:
            addr = f"{addr}, postcode {postcode}"
        #addr = addr.append("postcode " + str(getattr(r, "postcode", "unknown"))) if hasattr(r, "postcode") else addr


        # # ---- nearby POIs ----
        # nearby_raw = getattr(r, "nearby_places", None)
        # if isinstance(nearby_raw, str):
        #     try:
        #         nearby = ast.literal_eval(nearby_raw)
        #     except (ValueError, SyntaxError):
        #         nearby = []
        # elif isinstance(nearby_raw, list):
        #     nearby = nearby_raw
        # else:
        #     nearby = []

        # nearby_str = "; ".join(nearby[:5]) if nearby else "no nearby places listed"

        # ---- nearby POIs ----
        nearby_raw = getattr(r, "nearby_places", None)

        if isinstance(nearby_raw, list):
        # legacy case: list of strings
            nearby_str = "; ".join(nearby_raw[:5])
        elif isinstance(nearby_raw, str):
         # current correct case: already a string
            nearby_str = nearby_raw
        else:
            nearby_str = "no nearby places listed"


        # ---- mode & duration ----
        mode = str(getattr(r, "mode", "unknown")).lower()
        dur_val = getattr(r, duration_col, 0) if duration_col else 0
        dur = int(round(float(dur_val))) if pd.notna(dur_val) else 0

        # ---- final sentence ----
        toks.append(
            f"At {hhmm}, on weekday {dow} (hour block {hb}), "
            f"the user was at coordinates ({lat}, {lon}). "
            f"The address is {addr}. "
            f"The transport mode was {mode}, with an activity duration of {dur} minutes. "
            f"Nearby places include: {nearby_str}."
        )

    return toks


In [ ]:
toks = tokens_compact_1week(
    df_u=sp_sampled2,
    max_events=40,
    prec=4
)

print(toks[:8])

['Date: 2019-09-02', 'At 20:21, on weekday 0 (hour block 20), the user was at coordinates (46.936, 7.4314). The address is Chutzenstrasse, Bern, postcode 3007. The transport mode was walk, with an activity duration of 503 minutes. Nearby places include: 0.054 km to the north-west: Residential Bahnhof Weissenbühl; 0.06 km to the north-east: Shopping Nido; 0.062 km to the west: Shopping; 0.062 km to the west: Shopping; 0.064 km to the west: Transportation Weissenbühl Bahnhof.', 'Date: 2019-09-03', 'At 05:24, on weekday 1 (hour block 5), the user was at coordinates (47.2148, 7.5201). The address is Vogelherdstrasse, Roamer, postcode 4513. The transport mode was car, with an activity duration of 615 minutes. Nearby places include: 0.082 km to the south-west: Residential Restaurant Industrie; 0.082 km to the south-west: Residential Restaurant Industrie; 0.082 km to the south-west: Residential Restaurant Industrie; 0.082 km to the south-west: Residential Restaurant Industrie; 0.088 km to the

### Update prompts Version 3 28 Jan 2026

In [ ]:
# ---------------------------
# Build prompts.  update on 28 Jan 2026 
# ------------------------

from textwrap import dedent

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
sp_prompt = sp_sampled2.copy()
sp_prompt["date_str"] = (
    pd.to_datetime(sp_prompt["started_at"], errors="coerce")
      .dt.tz_localize(None)
      .dt.date
      .astype(str)
)
print(sp_prompt[["started_at", "date", "date_str"]].head(5))

# build valid user-day table from EXISTING rows only
user_day = (
    sp_prompt
    .dropna(subset=["started_at"])
    .loc[:, ["user_id", "date_str"]]
    .drop_duplicates()
)
user_day = (
    user_day
    .groupby("user_id", group_keys=False)
    #.sample(n=2, random_state=RANDOM_SEED)
    #.rename(columns={"date_str": "sample_date"})
)

#print("user_day size:", len(user_day))   # MUST be > 0

rows_prompts = []

user_dates = (
    sp_prompt
    .dropna(subset=["started_at"])
    .loc[:, ["user_id", "date_str"]]
    .drop_duplicates()
    .groupby("user_id")["date_str"]
    .apply(list)
)

print(f"user_dates sample: {len(user_dates)}")

for user_id, dates in user_dates.items():
    
    df_u = sp_prompt[

        (sp_prompt["user_id"] == user_id) &
        (sp_prompt["date_str"].isin(dates))
    ]
    print(f"Processing user {user_id} for dates {dates}, df_u shape: {df_u.shape}")


    toks = tokens_compact_1week(df_u, max_events=MAX_EVENTS, prec=PREC)
    if not toks:
        continue
    prompt = (
        f"User: {user_id}\n\n"
        f"Mobility record for one week in Switzerland:\n"
        "Task:\n"
        "Based only on the mobility behaviour and nearby spatial context described below,"
        "infer the user's most likely household income level based on swiss census data and situation\n\n"
        "Please choose exactly one of the following categories (monthly household income in CHF):\n"
        "<4000, 4001-8000, 8001-12000, 12001-16000, 16001+.\n\n"
        "Guidelines:\n"
        "-Use only the information explicitly provided in the mobility record.\n"
        "- Base your reasoning on mobility intensity, locations visited, transport modes, and activity patterns.\n"
        "- Do not assume any unobserved personal attributes.\n"
        "- keep a neutral and factual tone.\n\n"
        "Output format:\n"
        "1. A brief rationale (no more than 200 words).\n"
        "2. One final line in strict JSON format, for example:\n"
        " household_income_level: 4001-8000.\n\n"
        "Mobility sequence:\n"
        + "\n".join(toks)
    )
    
    # store results as a dictionary
    rows_prompts.append({
        "user_id": user_id,
        "date": dates,
        "prompt": prompt})

PROMPTS_OUT = Path("/data/baliu/python_code/data/prompts_v3_1week_income_29Jan2026.json")

PROMPTS_OUT.parent.mkdir(parents=True, exist_ok=True)
with open(PROMPTS_OUT, "w", encoding="utf-8") as f:
    for r in rows_prompts:
        f.write(r["prompt"])
        f.write("\n\n" + "="*80 + "\n\n")

print(f"Saved {len(rows_prompts)} prompts -> {PROMPTS_OUT}")

# print sample
for r in rows_prompts[:1]:
    print("\n--- Sample Prompt ---")
    print(r["prompt"][:1000] + "\n...")   

                        started_at       date    date_str
0 2019-10-22 09:39:28.773999+00:00 2019-10-22  2019-10-22
1 2019-10-23 06:35:32.236999+00:00 2019-10-23  2019-10-23
2 2019-10-23 09:02:58.773999+00:00 2019-10-23  2019-10-23
3 2019-10-23 09:53:18.151999+00:00 2019-10-23  2019-10-23
4 2019-10-23 16:44:43.608999+00:00 2019-10-23  2019-10-23
user_dates sample: 2102
Processing user AAGAF for dates ['2019-10-22', '2019-10-23', '2019-10-24', '2019-10-25', '2019-10-26', '2019-10-27', '2019-10-28'], df_u shape: (23, 26)
Processing user AAINS for dates ['2019-11-13', '2019-11-14', '2019-11-15', '2019-11-16', '2019-11-17', '2019-11-18', '2019-11-19'], df_u shape: (34, 26)
Processing user AAQME for dates ['2019-12-10', '2019-12-11', '2019-12-12', '2019-12-13', '2019-12-14', '2019-12-15', '2019-12-16'], df_u shape: (23, 26)
Processing user AARWP for dates ['2019-09-18', '2019-09-19', '2019-09-20', '2019-09-21', '2019-09-22', '2019-09-23', '2019-09-24'], df_u shape: (35, 26)
Processing user 

Clean the prompts 

In [ ]:
import json
import re
from pathlib import Path

IN_PATH  = Path("/data/baliu/python_code/data/prompts_v3_1week_income_29Jan2026.json")
OUT_PATHh = Path("/data/baliu/python_code/data/prompts_v3_1week_income_29Jan2026_clean.txt")

def clean_prompt_text(text: str) -> str:
    """
    Remove markdown headers, separators, examples, and keep
    only instruction + mobility evidence.
    """
    # 1. remove separator lines ---
    text = re.sub(r"^\s*---\s*$", "", text, flags=re.MULTILINE)

    # 2. remove markdown headers like ## Task Overview
    text = re.sub(r"^\s*## .*?$", "", text, flags=re.MULTILINE)

    # 3. remove Example block
    text = re.sub(
        r'## Example:.*?---',
        '',
        text,
        flags=re.DOTALL
    )

    # 4. collapse multiple blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


cleaned_rows = []

with open(IN_PATH, "r", encoding="utf-8") as f:
    raw = f.read()

blocks = [b.strip() for b in re.split(r"\n{2,}={80,}\n{2,}", raw) if b.strip()]

# apply cleaning
cleaned_blocks = [clean_prompt_text(b) for b in blocks if clean_prompt_text(b)]

# write cleaned jsonl
with open(OUT_PATHh, "w", encoding="utf-8") as f:
    for b in cleaned_blocks:
        f.write(b)
        f.write("\n\n"+ "="*80 + "\n\n")

print(f"Cleaned prompts saved: {len(cleaned_blocks)} -> {OUT_PATHh}")

Cleaned prompts saved: 2102 -> /data/baliu/python_code/data/prompts_v3_1week_income_29Jan2026_clean.txt


## 3. Predict Sociodemographics using "gpt-oss-20b" model based on the prompts we just generated 

In [ ]:
import ollama

models = ollama.list()
print(models)

models=[Model(model='gpt-oss:20b', modified_at=datetime.datetime(2025, 11, 9, 20, 40, 9, 692297, tzinfo=TzInfo(3600)), digest='17052f91a42e97930aa6e28a6c6c06a983e6a58dbb00434885a0cf5313e376f7', size=13793441244, details=ModelDetails(parent_model='', format='gguf', family='gptoss', families=['gptoss'], parameter_size='20.9B', quantization_level='MXFP4'))]


In [ ]:
import os, json, re, subprocess, sys
import torch, requests
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    p = torch.cuda.get_device_properties(0)
    print("VRAM (GB):", round(p.total_memory/1024**3, 2))

# Check ollama is reachable
r = requests.get("http://127.0.0.1:11434/api/tags", timeout=3)
print("Ollama OK. Models:", [m["name"] for m in r.json().get("models",[])])

CUDA available: True
GPU: NVIDIA L4
VRAM (GB): 22.06
Ollama OK. Models: ['gpt-oss:20b']


read prompts from text version

In [ ]:
import json
import time
import os
import re
import sys
import signal
from pathlib import Path
import ollama
import pandas as pd

# --------------------------------------
# Config
# --------------------------------------
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"
client = ollama.Client(host=os.environ["OLLAMA_HOST"])


# prompts_v3_1week_income_29Jan2026_clean  prompts_v3_1week_income_29Jan2026
PROMPT_TXT = Path("/data/baliu/python_code/data/prompts_v3_1week_income_29Jan2026_clean.txt")
PRED_JSON  = Path("/data/baliu/python_code/data/preds_v3_1week_income_29Jan2026.jsonl")
PRED_CSV   = Path("/data/baliu/python_code/data/preds_v3_1week_income_29Jan2026.csv")

MODEL_NAME = "gpt-oss:20b"
SEP = "=" * 80

# ---- safety knobs ----
MAX_PROMPT_CHARS = 20_000     # prompt length guard
MAX_PROMPT_CHARS = 30_000     # prompt length guard, we can try longer prompts for version 2
TIMEOUT_SEC      = 600        # generation timeout per user
COOLDOWN_EVERY   = 20         # periodic cooldown interval
COOLDOWN_SEC     = 10

# --------------------------------------
# Timeout handling (linux)
# --------------------------------------
class TimeoutError(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutError("Generation timed out")

signal.signal(signal.SIGALRM, timeout_handler)

# --------------------------------------
# load done users
# --------------------------------------
def load_done_users(path: Path) -> set:
    done = set()
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    obj = json.loads(line)
                    if "user_id" in obj:
                        done.add(obj["user_id"])
                except Exception:
                    continue
    return done

# --------------------------------------
# Load prompts
# --------------------------------------
with open(PROMPT_TXT, "r", encoding="utf-8") as f:
    raw = f.read()

prompts = [p.strip() for p in raw.split(SEP) if p.strip()]
print(f"📦 Loaded {len(prompts)} prompts")

# --------------------------------------
# Load checkpoint once
# --------------------------------------
done_users = load_done_users(PRED_JSON)
print(f"🔁 Found {len(done_users)} completed users")

preds = []

# --------------------------------------
# Main loop
# --------------------------------------
try:
    for i, prompt in enumerate(prompts, 1):

        m = re.search(r"User:\s*(\S+)", prompt)
        if not m:
            print("⚠️ Cannot find user_id, skipping")
            continue

        user_id = m.group(1)

        if user_id in done_users:
            print(f"⏭️ Skipping {user_id}")
            continue

        print(f"\n🔮 [{i}/{len(prompts)}] Predicting {user_id}")

        # ---- prompt length guard ----
        if len(prompt) > MAX_PROMPT_CHARS:
            print(f"⚠️ Prompt too long ({len(prompt)} chars), skip {user_id}")
            continue

        try:
            print("💓 Heartbeat: sending to model")
            start_t = time.time()

            signal.alarm(TIMEOUT_SEC)

            resp = client.generate(
                model=MODEL_NAME,
                prompt=prompt,
                options={
                    "temperature": 0.1,
                    "num_ctx": 2048
                }
            )

            signal.alarm(0)

            dt = time.time() - start_t
            print(f"⏱️ Model returned in {dt:.1f}s")

            content = resp.get("response", "").strip()

            try:
                result = json.loads(content)
            except Exception:
                result = {
                    "raw_output": content,
                    "error": "json_parse_failed"
                }

            result["user_id"] = user_id
            preds.append(result)

            with open(PRED_JSON, "a", encoding="utf-8") as f:
                f.write(json.dumps(result, ensure_ascii=False) + "\n")
                f.flush()
                os.fsync(f.fileno())

            done_users.add(user_id)
            print(f"✅ Completed {user_id}")

        except TimeoutError:
            print(f"⏰ Timeout for {user_id}, skipping")
            continue

        except Exception as e:
            print(f" Error for {user_id}: {e}")
            time.sleep(2)
            continue            

        # ---- periodic cooldown ----
        if i % COOLDOWN_EVERY == 0:
            print("🧹 Cooling down GPU / Ollama...")
            time.sleep(COOLDOWN_SEC)

except KeyboardInterrupt:
    print("\n🛑 Interrupted by user. Progress saved safely.")
    sys.exit(0)

# --------------------------------------
# Save CSV snapshot
# --------------------------------------
if PRED_JSON.exists():
    df = pd.read_json(PRED_JSON, lines=True)
    df.to_csv(PRED_CSV, index=False)
    print(f"\n🎉 Saved {len(preds)} new predictions → {PRED_CSV}")
else:
    print("⚠️ No new predictions generated")


📦 Loaded 2102 prompts
🔁 Found 1421 completed users
⏭️ Skipping AAGAF
⏭️ Skipping AAINS
⏭️ Skipping AAQME
⏭️ Skipping AARWP
⏭️ Skipping ABARC
⏭️ Skipping ABECR
⏭️ Skipping ABENL
⏭️ Skipping ABISR
⏭️ Skipping ABLPH
⏭️ Skipping ACSUF
⏭️ Skipping ACWVM
⏭️ Skipping ADCMH
⏭️ Skipping AEBWY
⏭️ Skipping AEGHJ
⏭️ Skipping AEWVV
⏭️ Skipping AFNCT
⏭️ Skipping AFUVG
⏭️ Skipping AGHUY
⏭️ Skipping AHTYK
⏭️ Skipping AJJJR
⏭️ Skipping AJJNX
⏭️ Skipping AJKBY
⏭️ Skipping AJQGV
⏭️ Skipping AKGPG
⏭️ Skipping AKMIL
⏭️ Skipping AKOZE
⏭️ Skipping AKQKS
⏭️ Skipping ALKPX
⏭️ Skipping ALSMT
⏭️ Skipping AMGUM
⏭️ Skipping AMGZL
⏭️ Skipping AMYMD
⏭️ Skipping ANEFF
⏭️ Skipping ANMKC
⏭️ Skipping ANQHI
⏭️ Skipping AOVGU
⏭️ Skipping AOVLF
⏭️ Skipping APKPN
⏭️ Skipping APWYR
⏭️ Skipping AQFEA
⏭️ Skipping AQQGE
⏭️ Skipping AQREK
⏭️ Skipping ARFNG
⏭️ Skipping ARMZF
⏭️ Skipping ARZDI
⏭️ Skipping ASUKL
⏭️ Skipping ASXOD
⏭️ Skipping ATBSI
⏭️ Skipping ATCBP
⏭️ Skipping ATEAN
⏭️ Skipping ATKDY
⏭️ Skipping AUXLS
⏭️ Skipping A

SystemExit: 0

/home/baliu/.venvs/gptoss-env/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


: 

## Evaluation

In [ ]:
import pandas as pd
import json
import re
from pathlib import Path

# --------------------------------------
# Load raw predictions
# --------------------------------------
raw_path = Path("/data/baliu/python_code/data/preds_v3_1week_income_29Jan2026.jsonl")

#preds_v3_1week_income_29Jan2026

df = pd.read_json(raw_path, lines=True)

print(f"Loaded raw predictions: {len(df)} rows")

# --------------------------------------
# Income extraction functions
# --------------------------------------
def extract_from_text(text):
    if pd.isna(text):
        return None

    try:
        data = json.loads(text)
        income_level = data.get("income_level", "")
        if income_level:
            if income_level in [">=16000", ">=16001", "16001+", "16000+"]:
                return ">16000"
            return income_level
    except Exception:
        pass

    text = str(text).lower()

    norm = (
        text.lower()
            .replace("chf", "")
            .replace("\u202f", "")
            .replace("\xa0", "")
            .replace(",", "")
    )

    if re.search(
        r'(>=\s*16000|>=\s*16001|16001\+|16000\+|'
        r'more\s+than\s+16000|above\s+16000|at\s+least\s+16000)',
        norm
    ):
        return ">16000"

    m = re.search(r'between\s*(\d{3,5})\s*and\s*(\d{3,5})', norm)
    if m:
        return map_income_bin(int(m.group(1)), int(m.group(2)))

    m = re.search(r'(\d{1,2}[,.]?\d{3})\s*[-–]\s*(\d{1,2}[,.]?\d{3})', norm)

    if m:
        return map_income_bin(int(m.group(1)), int(m.group(2)))

    for g in ["<4000", "4001-8000", "8001-12000", "12001-16000", ">16000"]:
        if g in text:
            return g

    return None

def map_income_bin(low, high):
    if high <= 4000:
        return "<4000"
    elif low >= 4001 and high <= 8000:
        return "4001-8000"
    elif low >= 8001 and high <= 12000:
        return "8001-12000"
    elif low >= 12001 and high <= 16000:
        return "12001-16000"
    elif high >= 16000:
        return ">16000"
    else:
        return None

# --------------------------------------
# Apply extraction
# --------------------------------------
TEXT_COL = "raw_output"
df["income_level"] = df[TEXT_COL].apply(extract_from_text)

print(df.columns.tolist())
print(df.head(3))


print(df["income_level"].value_counts(dropna=False))

# --------------------------------------
# Save cleaned CSV
# --------------------------------------
#out_path = Path("/data/baliu/python_code/data/preds_v3_1week_income_29Jan2026_clean.jsonl")

df_clean = (
    df[["user_id", "income_level"]]
    .drop_duplicates(subset=["user_id"])
    .sort_values("user_id")
    .reset_index(drop=True)
)

out_path = Path("/data/baliu/python_code/data/preds_v3_1week_income_29Jan2026_clean.jsonl")
df_clean.to_json(out_path, orient="records", lines=True)
out_path = Path("/data/baliu/python_code/data/preds_v3_1week_income_29Jan2026_clean.csv")
df_clean.to_csv(out_path, index=False)


df_clean.to_csv(out_path, index=False)

print(f"✅ Clean predictions saved to: {out_path}")
print(f"📊 Rows: {len(df_clean)}")
print(df_clean.head(20))


Loaded raw predictions: 692 rows
['raw_output', 'error', 'user_id', 'prediction', 'explanation', 'household_income', 'income_level']
                                          raw_output              error  \
0  The user’s mobility pattern shows a high level...  json_parse_failed   
1  The user’s mobility pattern shows a typical co...  json_parse_failed   
2  The user’s mobility pattern shows a typical su...  json_parse_failed   

  user_id prediction explanation household_income income_level  
0   AAGAF        NaN         NaN              NaN  12001-16000  
1   AAINS        NaN         NaN              NaN    4001-8000  
2   AAQME        NaN         NaN              NaN   8001-12000  
income_level
8001-12000     325
12001-16000    143
>16000          81
None            74
4001-8000       53
<4000           16
Name: count, dtype: int64
✅ Clean predictions saved to: /data/baliu/python_code/data/preds_v3_1week_income_29Jan2026_clean.csv
📊 Rows: 692
   user_id income_level
0    AAGAF  1200

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------
# File paths
# --------------------------------------
gt_path = Path("/data/baliu/python_code/data/gt_with_age_household_income_level.csv")
pred_path = Path("/data/baliu/python_code/data/preds_v3_1week_income_29Jan2026_clean.csv")

# --------------------------------------
# Load data
# --------------------------------------
gt = pd.read_csv(gt_path)
pred = pd.read_csv(pred_path)

print(f"Loaded ground truth ({len(gt)} rows)")
print(f"Loaded predictions  ({len(pred)} rows)")

# --------------------------------------
# Select & align columns
# --------------------------------------
gt = gt[["user_id", "age_group", "household_size_group", "income_level"]]
# pred = pred[["user_id", "age_group", "household_sizes", "income_level"]]
pred = pred[["user_id", "income_level"]]

# --------------------------------------
# Normalize household size
# --------------------------------------
# def normalize_household_size(x):
#     if pd.isna(x):
#         return np.nan
#     try:
#         x = float(x)
#     except Exception:
#         return np.nan

#     if x >= 5:
#         return "5+"
#     else:
#         return str(int(x))

# gt["hh_norm"] = gt["household_size_group"].apply(normalize_household_size)
# pred["hh_norm"] = pred["household_sizes"].apply(normalize_household_size)

# --------------------------------------
# Merge GT and predictions
# --------------------------------------
merged = pd.merge(
    gt,
    pred,
    on="user_id",
    suffixes=("_true", "_pred")
)

print(f"Merged {len(merged)} users")

# --------------------------------------
# Masks for valid evaluation samples
# --------------------------------------
# age_mask = (
#     merged["age_group_true"].notna() &
#     merged["age_group_pred"].notna()
# )

# hh_mask = (
#     merged["hh_norm_true"].notna() &
#     merged["hh_norm_pred"].notna()
# )

income_mask = (
    merged["income_level_true"].notna() &
    merged["income_level_pred"].notna() &
    (merged["income_level_true"].str.lower() != "prefer not to say")
)

print("\nEvaluation sample sizes:")
# print(f"Age:        {age_mask.sum()}")
# print(f"Household:  {hh_mask.sum()}")
print(f"Income:     {income_mask.sum()}")

# --------------------------------------
# Accuracy computation
# # --------------------------------------
# age_acc = (
#     merged.loc[age_mask, "age_group_true"]
#     == merged.loc[age_mask, "age_group_pred"]
# ).mean()

# household_acc = (
#     merged.loc[hh_mask, "hh_norm_true"]
#     == merged.loc[hh_mask, "hh_norm_pred"]
# ).mean()

income_acc = (
    merged.loc[income_mask, "income_level_true"]
    == merged.loc[income_mask, "income_level_pred"]
).mean()

# --------------------------------------
# Majority-class baselines (for context)
# --------------------------------------
def majority_baseline(y_true):
    majority = y_true.value_counts().idxmax()
    return (y_true == majority).mean()

# age_baseline = majority_baseline(
#     merged.loc[age_mask, "age_group_true"]
# )

# hh_baseline = majority_baseline(
#     merged.loc[hh_mask, "hh_norm_true"]
#)

# income_baseline = majority_baseline(
#     merged.loc[income_mask, "income_level_true"]
# )

# --------------------------------------
# Output results
# --------------------------------------
print("\n📊 Accuracy Results:")
# print(f"Age group accuracy:        {age_acc:.3f} (baseline {age_baseline:.3f})")
# print(f"Household size accuracy:  {household_acc:.3f} (baseline {hh_baseline:.3f})")
#print(f"Income level accuracy:    {income_acc:.3f} (baseline {income_baseline:.3f})")
print(f"Income level accuracy:    {income_acc:.3f} ")
# --------------------------------------
# Save merged comparison tables
# --------------------------------------
output_path = Path("/data/baliu/python_code/data/merged_comparison_v5_final.csv")
merged.to_csv(output_path, index=False, encoding="utf-8")

filtered_income_path = Path(
    "/data/baliu/python_code/data/comparison_income_eval_v5_final.csv"
)
merged.loc[income_mask].to_csv(
    filtered_income_path, index=False, encoding="utf-8"
)

print(f"\nDetailed comparison saved to: {output_path}")
print(f"Income-eval subset saved to: {filtered_income_path}")

# --------------------------------------
# (Optional) Confusion matrices
# --------------------------------------
# age_cm = pd.crosstab(
#     merged.loc[age_mask, "age_group_true"],
#     merged.loc[age_mask, "age_group_pred"],
#     normalize="index"
# )

# age_cm_path = Path(
#     "/data/baliu/python_code/data/age_confusion_matrix_v4_final.csv"
# )
# age_cm.to_csv(age_cm_path)

# print(f"Age confusion matrix saved to: {age_cm_path}")

# ---- Confusion matrix (income) ----
income_cm = pd.crosstab(
    merged.loc[income_mask, "income_level_true"],
    merged.loc[income_mask, "income_level_pred"]
    #normalize="index"
)

print("\nIncome confusion matrix (counts):")
print(income_cm)

income_cm = pd.crosstab(
    merged.loc[income_mask, "income_level_true"],
    merged.loc[income_mask, "income_level_pred"],
    normalize="index"
)

print("\nIncome confusion matrix (row-normalized):")
print(income_cm)

/tmp/ipykernel_34707/1873157119.py:14: DtypeWarning: Columns (4,6,7,8,9,10,11,12,13,15,17,18,19,20,21,22,23,28,29,30,31,32,33,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,113,114,115,116,117,118,119,120,121,122,123) have mixed types. Specify dtype option on import or set low_memory=False.
  gt = pd.read_csv(gt_path)


Loaded ground truth (90909 rows)
Loaded predictions  (692 rows)
Merged 623 users

Evaluation sample sizes:
Income:     495

📊 Accuracy Results:
Income level accuracy:    0.269 

Detailed comparison saved to: /data/baliu/python_code/data/merged_comparison_v5_final.csv
Income-eval subset saved to: /data/baliu/python_code/data/comparison_income_eval_v5_final.csv

Income confusion matrix (counts):
income_level_pred  12001-16000  4001-8000  8001-12000  <4000  >16000
income_level_true                                                   
12001-16000                 20          7          39      0       8
4001-8000                   34         16          79      2      30
8001-12000                  35         11          90      6      21
<4000                        5          5          30      1       4
>16000                      19          1          25      1       6

Income confusion matrix (row-normalized):
income_level_pred  12001-16000  4001-8000  8001-12000     <4000    >16000
inc

: 